In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import timm
from tqdm import tqdm

c:\Users\Divyaa Sriram\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Enable faster CUDA operations
torch.backends.cudnn.benchmark = True  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# 🔹 Define transformations (Xception requires 299x299 images)
transform = transforms.Compose([
    transforms.Resize((299, 299)),  # XceptionNet expects 299x299 input
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Normalization for Xception
])

In [4]:
# 🔹 Load datasets
train_dir = "D:/Projects/Minor Project/Deepfake Detection/datasets/New folder/real-vs-fake/train"  # Update with correct path
val_dir = "D:/Projects/Minor Project/Deepfake Detection/datasets/New folder/real-vs-fake/valid"  # Update with correct path

In [5]:
try:
    train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
    val_dataset = datasets.ImageFolder(root=val_dir, transform=transform)
    print(f"✅ Train Dataset: {len(train_dataset)} images")
    print(f"✅ Validation Dataset: {len(val_dataset)} images")
except Exception as e:
    print(f"❌ Error loading datasets: {e}")

✅ Train Dataset: 100000 images
✅ Validation Dataset: 20000 images


In [6]:
# 🔹 Optimized Data Loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

In [7]:
# 🔹 Load pretrained XceptionNet model
model = timm.create_model("xception", pretrained=True)  # Load XceptionNet

c:\Users\Divyaa Sriram\AppData\Local\Programs\Python\Python311\Lib\site-packages\timm\models\_factory.py:126: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to C:\Users\Divyaa Sriram/.cache\torch\hub\checkpoints\xception-43020ad28.pth


In [8]:
# 🔹 Modify classifier for binary classification (Real vs Fake)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)  # 2 classes: Real & Deepfake
model = model.to(device)

In [9]:
# 🔹 Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# 🔹 Enable mixed precision for faster training
scaler = torch.cuda.amp.GradScaler()

c:\Users\Divyaa Sriram\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\amp\grad_scaler.py:126: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [10]:
# 🔹 Training loop with validation
num_epochs = 5
best_acc = 0.0  # Track best validation accuracy

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for inputs, labels in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        # Enable mixed precision training
        with torch.cuda.amp.autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        # Backpropagation with GradScaler
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Prevent exploding gradients
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

        progress_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{(correct/total)*100:.2f}%")

Epoch 1/5:   0%|          | 0/6250 [00:00<?, ?it/s]c:\Users\Divyaa Sriram\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\amp\autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Epoch 1/5:  44%|████▍     | 2754/6250 [7:56:54<10:05:24, 10.39s/it, acc=97.18%, loss=0.0034]


KeyboardInterrupt: 

In [ ]:
    # 🔹 Calculate epoch loss and accuracy
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    print(f"🔹 Epoch {epoch+1}: Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.2f}%")

    # 🔹 **Validation Loop**
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for val_inputs, val_labels in val_loader:
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)

            with torch.cuda.amp.autocast():
                val_outputs = model(val_inputs)
            _, val_preds = torch.max(val_outputs, 1)

            val_correct += torch.sum(val_preds == val_labels).item()
            val_total += val_labels.size(0)

    val_acc = (val_correct / val_total) * 100
    print(f"✅ Validation Accuracy: {val_acc:.2f}%")

In [ ]:
    # 🔹 Save best model based on validation accuracy
    if val_acc > best_acc:
        best_acc = val_acc
        model_path = "best_xception_model.pth"  # Save in local directory
        torch.save(model.state_dict(), model_path)
        print(f"🎉 Best model saved at epoch {epoch+1} with validation accuracy {best_acc:.2f}%")

print("✅ Training complete!")